# Day 040 Project: Auto-Generated Data Story

## What You're Building

A full automated data story pipeline: load a CSV, run EDA, narrate individual columns and groups, then generate an executive summary — all driven by Ollama.

**Deliverable:** You run every cell top-to-bottom. The final checks pass. You have a printed narrative that a non-technical stakeholder could read.

## Project Requirements

1. Load `RETAIL_CSV` into a DataFrame with a `revenue` column
2. Narrate the `revenue` column using `summarize_column`
3. Narrate the top products using `narrate_top_groups`
4. Narrate the correlations with `revenue` using `narrate_correlations`
5. Generate a full executive summary using `generate_data_story`
6. Verify with `_run_project_checks()`

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import json
import ollama
import pandas as pd
import io


def distribution_summary(df, col):
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        return {
            'count': int(s.count()), 'mean': round(float(s.mean()), 4),
            'std': round(float(s.std()), 4), 'min': float(s.min()),
            'q25': float(s.quantile(0.25)), 'median': float(s.quantile(0.50)),
            'q75': float(s.quantile(0.75)), 'max': float(s.max()),
            'null_count': int(s.isnull().sum()),
        }
    counts = s.value_counts()
    return {
        'count': int(s.count()), 'unique': int(s.nunique()),
        'top': str(counts.index[0]) if len(counts) else None,
        'top_freq': int(counts.iloc[0]) if len(counts) else 0,
        'null_count': int(s.isnull().sum()),
    }


def top_groups(df, group_col, value_col, n=5):
    return (
        df.groupby(group_col)[value_col]
        .agg(total='sum', mean='mean', count='count')
        .reset_index()
        .nlargest(n, 'total')
        .reset_index(drop=True)
    )


def correlation_summary(df, target_col):
    corr = df.select_dtypes(include='number').corr()[target_col].drop(target_col)
    result = pd.DataFrame({'feature': corr.index.tolist(), 'correlation': corr.values})
    result['_abs'] = result['correlation'].abs()
    result = result.sort_values('_abs', ascending=False).drop(columns='_abs')
    return result.reset_index(drop=True)


def eda_report(df):
    num_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    return {
        'shape':           df.shape,
        'null_counts':     df.isnull().sum().to_dict(),
        'numeric_summary': df[num_cols].describe().round(2).to_dict() if num_cols else {},
        'category_counts': {col: df[col].value_counts().to_dict() for col in cat_cols},
        'correlations':    df[num_cols].corr().round(4).to_dict() if len(num_cols) > 1 else {},
    }


import json
import ollama

def summarize_column(col_name: str, stats: dict,
                     model: str = 'llama3.2') -> str:
    prompt = (
        f"You are a concise data analyst. Describe the column '{col_name}' "
        f"in 1-2 clear sentences for a non-technical reader.\n\n"
        f"Statistics:\n{json.dumps(stats, indent=2)}"
    )
    resp = ollama.chat(model=model,
                       messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"].strip()


import json
import ollama

def narrate_top_groups(groups_df, group_col: str, value_col: str,
                       model: str = 'llama3.2') -> str:
    records = groups_df.to_dict(orient='records')
    prompt = (
        f"You are a concise data analyst. Write 2-3 sentences about which "
        f"'{group_col}' groups have the highest '{value_col}' and what stands out.\n\n"
        f"Top groups by {value_col}:\n{json.dumps(records, indent=2)}"
    )
    resp = ollama.chat(model=model,
                       messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"].strip()


import json
import ollama

def narrate_correlations(corr_df, target_col: str,
                         model: str = 'llama3.2') -> str:
    records = corr_df.to_dict(orient='records')
    prompt = (
        f"You are a concise data analyst. Write 2-3 sentences explaining which "
        f"features correlate most with '{target_col}' and what this likely means.\n\n"
        f"Correlations with '{target_col}':\n{json.dumps(records, indent=2)}"
    )
    resp = ollama.chat(model=model,
                       messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"].strip()


import json
import ollama

def narrate_eda_report(report: dict, title: str = 'Dataset',
                       model: str = 'llama3.2') -> str:
    shape = report['shape']
    nulls = sum(report['null_counts'].values())
    summary = {
        'dataset':             title,
        'rows':                shape[0],
        'columns':             shape[1],
        'total_nulls':         nulls,
        'numeric_columns':     list(report['numeric_summary'].keys()),
        'categorical_columns': list(report['category_counts'].keys()),
        'numeric_stats':       report['numeric_summary'],
        'top_categories':      {
            col: dict(list(counts.items())[:5])
            for col, counts in report['category_counts'].items()
        },
    }
    prompt = (
        "You are a data analyst. Write a 3-5 sentence executive summary of this "
        "dataset for a business audience. Highlight key patterns, data quality, "
        "and notable findings.\n\n"
        f"EDA Report:\n{json.dumps(summary, indent=2, default=str)}"
    )
    resp = ollama.chat(model=model,
                       messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"].strip()


import ollama
import pandas as pd

def generate_data_story(df: pd.DataFrame, title: str = 'Dataset',
                        model: str = 'llama3.2') -> str:
    report = eda_report(df)
    return narrate_eda_report(report, title=title, model=model)


RETAIL_CSV = (
    'order_id,product,category,region,price,quantity\n'
    '1,Widget,Electronics,North,25.0,10\n'
    '2,Gadget,Electronics,South,150.0,3\n'
    '3,Widget,Electronics,South,25.0,5\n'
    '4,Doohickey,Accessories,East,8.0,50\n'
    '5,Gadget,Electronics,East,150.0,7\n'
    '6,Widget,Electronics,East,25.0,4\n'
    '7,Doohickey,Accessories,North,8.0,20\n'
    '8,Gadget,Electronics,North,150.0,2\n'
    '9,Widget,Electronics,West,25.0,6\n'
    '10,Doohickey,Accessories,South,8.0,15\n'
    '11,Thingamajig,Accessories,North,200.0,1\n'
    '12,Thingamajig,Accessories,East,200.0,4'
)
SALES_DF = pd.read_csv(io.StringIO(RETAIL_CSV))
SALES_DF['revenue'] = SALES_DF['price'] * SALES_DF['quantity']

print(f'Loaded {SALES_DF.shape[0]} rows')

## Your Data Story

In [ ]:
# Step 1: Narrate the revenue column
# TODO: rev_stats  = distribution_summary(SALES_DF, 'revenue')
# TODO: rev_narr   = summarize_column('revenue', rev_stats)
# TODO: print('Revenue column:\n', rev_narr)

# Step 2: Narrate top products by revenue
# TODO: top3       = top_groups(SALES_DF, 'product', 'revenue', n=4)
# TODO: group_narr = narrate_top_groups(top3, 'product', 'revenue')
# TODO: print('\nProduct rankings:\n', group_narr)

# Step 3: Narrate correlations with revenue
# TODO: corr       = correlation_summary(SALES_DF, 'revenue')
# TODO: corr_narr  = narrate_correlations(corr, 'revenue')
# TODO: print('\nCorrelations:\n', corr_narr)

# Step 4: Full executive summary
# TODO: story = generate_data_story(SALES_DF, title='Retail Sales Q4')
# TODO: print('\n=== Executive Summary ===\n', story)

## Project Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: SALES_DF has revenue column
    try:
        assert 'SALES_DF' in globals() and 'revenue' in SALES_DF.columns
        passed += 1; print(f'\u2705 Check 1: SALES_DF loaded ({len(SALES_DF)} rows)')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: rev_narr is a non-empty string
    try:
        assert 'rev_narr' in globals(), 'rev_narr not defined'
        assert isinstance(rev_narr, str) and len(rev_narr.strip()) > 0
        passed += 1; print(f'\u2705 Check 2: rev_narr is {len(rev_narr)} chars')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: group_narr is a non-empty string
    try:
        assert 'group_narr' in globals(), 'group_narr not defined'
        assert isinstance(group_narr, str) and len(group_narr.strip()) > 0
        passed += 1; print(f'\u2705 Check 3: group_narr is {len(group_narr)} chars')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: corr_narr is a non-empty string
    try:
        assert 'corr_narr' in globals(), 'corr_narr not defined'
        assert isinstance(corr_narr, str) and len(corr_narr.strip()) > 0
        passed += 1; print(f'\u2705 Check 4: corr_narr is {len(corr_narr)} chars')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: story is a substantial executive summary (> 100 chars)
    try:
        assert 'story' in globals(), 'story not defined'
        assert isinstance(story, str) and len(story.strip()) > 100, \
            f'story too short ({len(story)} chars)'
        passed += 1; print(f'\u2705 Check 5: story is {len(story)} chars (executive summary)')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Loop `summarize_column` over every column in `SALES_DF` and print each summary
- Ask the LLM to suggest one business action based on the data story
- Pass a system prompt to `ollama.chat` to set the analyst's persona (e.g. 'You are a CFO summarising sales data for the board')
- Try a different model (`model='mistral'` if installed) and compare the summaries
- On Day 50 you will build the Insight Engine capstone — upload any CSV and get an auto-generated story back via a web UI